# Track 1 — Smart Campus Analytics · Prediction Notebook
**Team kkboys**

Loads the trained model, rebuilds the training-time features, and writes one prediction per
test row in the input order, under the column `prediction`.

Designed for the evaluation sandbox: no internet, no `train.csv`, and a test file with no ID
column and shuffled rows. Nothing here reads the training data or fits anything — the model
and the imputer inside it are already fitted.

**Model:** Ridge regression on 370 building-aware features, with sub-model imputation
(a LightGBM per gap-prone column) and training-time missingness augmentation, averaged over
5 random seeds. Only libraries present in the platform image are used
(`numpy`, `pandas`, `scikit-learn`, `joblib`, `lightgbm`).

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from sklearn.base import BaseEstimator, TransformerMixin

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")
print(f"input : {TEST_PATH}")
print(f"output: {OUTPUT_PATH}")

## Feature engineering

Identical to the training notebook. The function is pure — it learns nothing from the data —
so test rows are transformed exactly as training rows were.

The per-building blocks give each building its own intercept, its own 24-hour load profile,
its own weekend offset, and its own slopes on occupancy, temperature and previous usage. The
`*_missing` flags and `n_missing` let the model know which values were observed rather than
estimated, which matters because the test set carries substantially more gaps than training did.

In [ ]:
BUILDINGS = ["ADM_A", "BUS_A", "BUS_B", "ENG_A", "ENG_B", "LEC_A",
             "LIB_A", "RES_A", "RES_B", "SCI_A", "SCI_B", "SPT_A"]
DOW_MAP = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
           "Friday": 4, "Saturday": 5, "Sunday": 6}
NUMERIC = ["temperature", "humidity", "occupancy", "previous_usage"]
HELPER  = ["hour", "month", "dow", "weekend", "building_code"]


def make_features(df):
    """The SAME feature engineering as in training."""
    df = df.copy()
    X = pd.DataFrame(index=df.index)

    # --- time, with cyclical encoding so hour 23 sits beside hour 0 ---------
    X["hour"]  = df["hour"]
    X["month"] = df["month"]
    X["dow"]   = df["day_of_week"].map(DOW_MAP)
    X["weekend"]   = (X["dow"] >= 5).astype(int)
    X["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    X["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
    X["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    X["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    # --- drivers, plus explicit missingness signals -------------------------
    for c in NUMERIC:
        X[c] = df[c]
        X[c + "_missing"] = df[c].isna().astype(int)
    X["n_missing"] = df[NUMERIC].isna().sum(axis=1)

    X["building_code"] = df["building_id"].map({b: i for i, b in enumerate(BUILDINGS)})

    # --- per-building intercept, daily profile, weekend offset, slopes ------
    extra = {}
    for b in BUILDINGS:
        is_b = (df["building_id"] == b).astype(float)
        extra[f"b_{b}"] = is_b
        extra[f"b_{b}_weekend"] = is_b * X["weekend"]
        for src, nm in [("occupancy", "occ"), ("temperature", "temp"), ("previous_usage", "prev")]:
            extra[f"b_{b}_{nm}"] = is_b * df[src]
        for h in range(24):
            extra[f"b_{b}_h{h}"] = is_b * (df["hour"] == h).astype(float)

    # --- temperature x time of day: cooling load depends on both jointly ----
    t = df["temperature"]
    extra["temp_x_hour_sin"] = t * X["hour_sin"]
    extra["temp_x_hour_cos"] = t * X["hour_cos"]
    extra["temp_sq"]    = t ** 2
    extra["temp_x_occ"] = t * df["occupancy"]

    return pd.concat([X, pd.DataFrame(extra, index=df.index)], axis=1)

## The imputer inside the model

This class must be defined before `joblib.load`, because the saved pipeline contains a fitted
instance of it and the pickle refers to it by name.

It is already fitted — nothing is learned here. Each gap-prone column has its own LightGBM that
predicts the missing value from building, hour and the remaining drivers. A median would be
actively harmful: the building slope terms are products `is_building × value`, so a single gap
collapses the building-specific relationship for that row.

In [ ]:
class SubModelImputer(BaseEstimator, TransformerMixin):
    """Fitted at training time; here it only transforms.

    Fills each gap with a per-column LightGBM prediction conditioned on building, hour
    and the other drivers, then regenerates the derived interaction terms from the
    filled values. The *_missing flags are left untouched so the downstream model can
    still tell an observed value from an estimated one.
    """

    def __init__(self, extra_frame=None):
        self.extra_frame = extra_frame

    def fit(self, X, y=None):
        src = X if self.extra_frame is None else pd.concat([X, self.extra_frame], ignore_index=True)
        self.models_, self.median_ = {}, {}
        for c in NUMERIC:
            self.median_[c] = src[c].median()
            observed = src[c].notna()
            feats = HELPER + [o for o in NUMERIC if o != c]
            if observed.sum() < 50:
                self.models_[c] = None
                continue
            m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                  random_state=42, verbose=-1, n_jobs=-1)
            m.fit(src.loc[observed, feats], src.loc[observed, c])
            self.models_[c] = (m, feats)
        return self

    def transform(self, X):
        X = X.copy()
        for c in NUMERIC:
            gaps = X[c].isna()
            if not gaps.any():
                continue
            if self.models_[c] is None:
                X.loc[gaps, c] = self.median_[c]
            else:
                m, feats = self.models_[c]
                tmp = X.loc[gaps, feats].copy()
                for col in feats:
                    if tmp[col].isna().any():
                        tmp[col] = tmp[col].fillna(self.median_.get(col, 0.0))
                X.loc[gaps, c] = m.predict(tmp)

        for b in BUILDINGS:
            if f"b_{b}" not in X.columns:
                continue
            is_b = X[f"b_{b}"]
            for src, nm in [("occupancy", "occ"), ("temperature", "temp"), ("previous_usage", "prev")]:
                if f"b_{b}_{nm}" in X.columns:
                    X[f"b_{b}_{nm}"] = is_b * X[src]
        if "temp_sq" in X.columns:
            X["temp_sq"]        = X["temperature"] ** 2
            X["temp_x_occ"]     = X["temperature"] * X["occupancy"]
            X["temp_x_hour_sin"] = X["temperature"] * X["hour_sin"]
            X["temp_x_hour_cos"] = X["temperature"] * X["hour_cos"]
        return X.fillna(0.0)

## Load, predict, write

`model.pkl` holds five fitted pipelines (one per random seed) and the exact feature list used
in training. Predictions are the mean across the five — averaging away the arbitrary choice of
augmentation seed.

In [ ]:
models, features = joblib.load("model.pkl")
print(f"loaded {len(models)} models, {len(features)} features")

test_df = pd.read_csv(TEST_PATH)
print(f"test rows: {len(test_df):,}  columns: {list(test_df.columns)}")

X_test = make_features(test_df)

missing = [c for c in features if c not in X_test.columns]
if missing:
    raise RuntimeError(f"feature mismatch — missing {len(missing)}: {missing[:5]}")
X_test = X_test[features]

preds = np.mean([m.predict(X_test) for m in models], axis=0)

In [ ]:
# --- validate before writing -------------------------------------------------
assert len(preds) == len(test_df), f"row count mismatch: {len(preds)} vs {len(test_df)}"
assert np.isfinite(preds).all(), "non-finite predictions"
if (preds < 0).any():
    n_neg = int((preds < 0).sum())
    print(f"clipping {n_neg} negative predictions to 0 (energy cannot be negative)")
    preds = np.clip(preds, 0, None)

pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")
print(pd.Series(preds).describe().round(3).to_string())